In [5]:
import pandas as pd
import urllib.request
import urllib.error
from bs4 import BeautifulSoup

In [6]:
headers = {'User-Agent':'Mozilla/5.0 (X11; CrOS x86_64 14541.0.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36'}

In [7]:
def scrape_schedule(url):

    """
    This is a quick module modeled on my home-ice scraper. The goal is to scrape the 2024-25 NHL schedule into a series of away-team,
    home-team tuples, or at least create a csv that my season simulation module can parse.
    """

    req = urllib.request.Request(url, headers=headers)
    # Error handling
    try:
        #html = urlopen(url)
        with urllib.request.urlopen(req) as response:
            html = response.read().decode('utf-8')
    except urllib.error.HTTPError as e:
        print(f"HTTP error {e} - could not retrieve data from {url}") # return null, break, or something else
        return pd.DataFrame()
    except urllib.error.URLError as e:
        print(f"URL error {e}. The server could not be found for {url}.")
        return pd.DataFrame()
    
    # the program continues. If I return or break in the exception catch, "else" isn't needed.

    soup = BeautifulSoup(html, 'html.parser')

    game_teams = []

    regular_season_table = None
    
    regular_season_table = soup.find('table', {'id': 'games'}) 

    # Iterate through each row in the table

    for row in regular_season_table.tbody.find_all('tr'):
        
        # Skip rows without data or that are headers

        if 'class' in row.attrs and 'thead' in row['class']:
            continue

        cols = row.find_all('td')
        #if len(cols) < 10:
        #    continue

        try:

            # Get the team codes for both teams
            vis_team_link = cols[1].find('a')['href']
            vis_team_parts = vis_team_link.split('/')
            vis_team_code = vis_team_parts[-2]
            
            home_team_link = cols[3].find('a')['href']
            home_team_parts = home_team_link.split('/')
            home_team_code = home_team_parts[-2]

            game_teams.append({
                'VISITOR': vis_team_code,
                'HOME': home_team_code
            })

                    
        except(ValueError, IndexError) as e:
            print(f'Skipping a row due to a data error: {e}')
            continue
    
    return pd.DataFrame(game_teams)

In [8]:
url = "https://www.hockey-reference.com/leagues/NHL_2025_games.html"

schedule = scrape_schedule(url)

In [9]:
schedule.head()

,VISITOR,HOME
0,NJD,BUF
1,BUF,NJD
2,BOS,FLA
3,STL,SEA
4,CHI,UTA


In [10]:
schedule.to_csv('NHL_schedule_2025.csv', index=False)

In [11]:
schedule

,VISITOR,HOME
0,NJD,BUF
1,BUF,NJD
2,BOS,FLA
3,STL,SEA
4,CHI,UTA
...,...,...
1307,CGY,LAK
1308,TBL,NYR
1309,CAR,OTT
1310,WSH,PIT
